# Quickstart - Diabetes Risk Prediction

**Author:** Rudolph Otoo  
**Run time:** ~30-60 seconds (Colab ready)

This notebook exercises **both pillars** of the repository in a single pass:

1. **Statistical ML** - train a logistic-regression risk classifier and report
   ROC-AUC / confusion matrix on the held-out test set.
2. **Mechanistic ODE** - simulate a 100-day **SEIRD** compartment model and plot
   the trajectories.

> Runs entirely in-browser via [Google Colab](https://colab.research.google.com),
> or locally with `pip install -r requirements.txt`.

## 0. Environment Setup

If run in **Colab**, the next cell clones the repository and installs the
scientific stack. If run locally from the repository root, it is a no-op.

In [ ]:
# Colab-aware setup (no-op when run locally from the repository root)
import os, sys, subprocess

REPO_URL = "https://github.com/rudolphOtoo/diabetes-risk-prediction.git"

if not os.path.isdir("src"):
    # Running in Colab: clone the repo and step into its root.
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    os.chdir("repo")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )

sys.path.insert(0, os.path.abspath("src"))

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import Settings
from src.data import process_data
from src.features import split_features_target
from src.tune import tune_model
from src.evaluate import evaluate_model
from src.models.ode import ODEModelConfig, solve_seird
from src.visualize import (
    set_global_style,
    plot_roc_curves,
    plot_confusion_matrices,
)

set_global_style()
settings = Settings()

## 1. Pillar 1 - Statistical ML (risk classification)

Train a single regularised **LogisticRegression** under the repository's
leakage-safe protocol (median imputation + standardisation **inside** the
pipeline, fitted on training folds only) and evaluate on the held-out test set.

In [ ]:
# Load data, split, and train one interpretable model
frame  = process_data()
splits = split_features_target(frame, settings=settings)

X_train, X_test = splits["X_train"], splits["X_test"]
y_train, y_test = splits["y_train"], splits["y_test"]

fitted, best_params = tune_model("logistic_regression", X_train, y_train, settings=settings)
print("Best params:", best_params)

metrics = evaluate_model(fitted, X_test, y_test, model_name="logistic_regression")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:>16}: {v:.4f}")
    else:
        print(f"  {k:>16}: {v}")

In [ ]:
# Inline ROC + confusion matrix
plot_roc_curves({"logistic_regression": fitted}, X_test, y_test, show=True)
plot_confusion_matrices({"logistic_regression": fitted}, X_test, y_test, show=True)
plt.show()

## 2. Pillar 2 - Mechanistic ODE (SEIRD simulation)

Simulate a deterministic **SEIRD** compartment system (Susceptible -> Exposed ->
Infectious -> Recovered/Deceased) over **100 days** in a closed population of
N = 100,000. The solver verifies the conservation law
(S + E + I + R + D = N) and non-negativity on every trajectory.

In [ ]:
# 100-day SEIRD simulation
config = ODEModelConfig(n_population=100_000, n_days=100, master_seed=42)
traj = solve_seird({}, config=config)

t  = traj["t"]
tot = sum(traj[k] for k in traj if k != "t")
print(f"Max conservation error (|sum - N|): {float(np.abs(tot - 100000).max()):.2e}")
print(f"Peak infectious: {traj['I'].max():,.0f} on day {t[int(traj['I'].argmax())]:.1f}")

In [ ]:
# Inline SEIRD trajectories
fig, ax = plt.subplots(figsize=(7, 5))
for name in ["S", "E", "I", "R", "D"]:
    ax.plot(t, traj[name], lw=2, label=name)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Compartment population")
ax.set_title("SEIRD Compartment Trajectories (100 days)")
ax.legend(frameon=True)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

You have just executed both modelling paradigms end-to-end:

| Pillar | Question answered | Method |
|---|---|---|
| Statistical ML | "Who is at risk?" | Logistic regression, leakage-safe pipeline, ROC-AUC ~ 0.8 |
| Mechanistic ODE | "How does prevalence evolve?" | Deterministic SEIRD ODE with conservation verification |

For the full academic rationale, see
[`docs/paradigm_justification.md`](../docs/paradigm_justification.md).